In [1]:
%run code/losses.py

In [2]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

# from losses import compute_fkl, compute_rkl
from datasets import load_from_disk

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 用于 On-Policy 生成
max_new_tokens = 128
temperature = 2.0


class OPDTrainer(SFTTrainer):

    def __init__(self, *args, teacher_model=None, max_new_tokens=128, temp=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.max_new_tokens = max_new_tokens
        self.temp = temp
        self.tokenizer = kwargs.get("processing_class", kwargs.get("tokenizer"))

    def _generate_on_policy(self, model, input_ids, attention_mask):
        was_training = model.training
        model.eval()
        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                top_p=0.9,
                temperature=0.8,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                use_cache=True,
            )
        if was_training:
            model.train()
        return generated_ids

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        prompt_input_ids = inputs["input_ids"]
        if "attention_mask" in inputs:
            prompt_attention_mask = inputs["attention_mask"]
        else:
            prompt_attention_mask = prompt_input_ids.ne(self.tokenizer.pad_token_id).long()
            
        prompt_lengths = prompt_attention_mask.sum(dim=1)

        # 1. 学生模型自行生成轨迹
        generated_ids = self._generate_on_policy(
            model, prompt_input_ids, prompt_attention_mask
        )
        generated_attention_mask = generated_ids.ne(self.tokenizer.pad_token_id).long()

        # 2. 构建 labels：把 prompt 部分的 token 设为 -100
        labels = generated_ids.clone()
        for row_idx, prompt_len in enumerate(prompt_lengths):
            labels[row_idx, :prompt_len] = -100
        labels = labels.masked_fill(generated_attention_mask.eq(0), -100)

        # 3. 学生模型前向传播（【核心机制】只拿 logits，坚决不计算自身 SFT Loss）
        outputs_student = model(
            input_ids=generated_ids,
            attention_mask=generated_attention_mask,
            return_dict=True
        )
        logits = outputs_student.logits

        with torch.no_grad():
            teacher_outputs = self.teacher_model(
                input_ids=generated_ids,
                attention_mask=generated_attention_mask,
                return_dict=True
            )
            teacher_logits = teacher_outputs.logits

        # 如果教师模型和学生模型输出形状不匹配，对教师模型进行截断
        kl = 0
        if isinstance(logits, torch.Tensor) and isinstance(teacher_logits, torch.Tensor):
            if logits.shape[-1] != teacher_logits.shape[-1]:
                teacher_logits = teacher_logits[:, :, :logits.shape[-1]]

            # kl = compute_fkl(logits, teacher_logits, labels, padding_id=-100, temp=2.0)
            kl = compute_rkl(logits, teacher_logits, labels, padding_id=-100, temp=self.temp)
            
            # 对 KL loss 进行有效 token 数量的平均（防止模型钻短序列的空子）
            valid_tokens = labels.ne(-100).sum().clamp_min(1)
            kl = kl / valid_tokens

        # GKD 范式：总损失 = 纯 KL 散度
        loss_total = kl

        return (loss_total, outputs_student) if return_outputs else loss_total

# 学生模型
student, _ = FastLanguageModel.from_pretrained(
    # Can select any from the below:
    # "unsloth/Qwen2.5-0.5B", "unsloth/Qwen2.5-1.5B", "unsloth/Qwen2.5-3B"
    # "unsloth/Qwen2.5-14B",  "unsloth/Qwen2.5-32B",  "unsloth/Qwen2.5-72B",
    # And also all Instruct versions and Math. Coding verisons!
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

student = FastLanguageModel.get_peft_model(
    student,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

student.print_trainable_parameters()

teacher, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "qwen_teacher_finetune",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(teacher)
teacher.eval() # 确保在训练过程中教师模型始终处于 eval 模式

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


# 读取数据集
# 直接读取 split_data.py 已经处理好格式的本地数据集
dataset = load_from_disk("./data_splits/data_full")

print(f"\n 成功加载本地数据集，数据量: {len(dataset)} 条")
print("【数据抽样检查 (应为不包含答案的 Prompt)】:\n", dataset[0]["text"])
print("-" * 50)


# 训练参数
args = TrainingArguments(
    output_dir='./results_gen_opd',
    num_train_epochs=4, # Baseline 全量跑 4 个 Epoch
    # max_steps=3, #测试用
    
    do_train=True,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    logging_steps=10,
    # logging_steps=1, #测试用

    save_strategy='no', # 最终结果统一保存
    bf16=True,
    learning_rate=0.0005,
    lr_scheduler_type='constant',
    optim="adamw_torch_fused",
    remove_unused_columns=False, #  必须关闭，否则会自动清除 input_ids 导致前向传播报错
)


trainer = OPDTrainer(
    model=student,
    teacher_model=teacher,

    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field = "text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Can make training 5x faster for short sequences.
    args=args,
    
    max_new_tokens=max_new_tokens,
    temp=temperature,
)


import warnings
import transformers
# 强行关闭 HuggingFace 生成时的各种警告
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

# 如果是初次训练resume_from_checkpoint为false，接着checkpoint继续训练，为True
print("\n 开始纯 KL 在线蒸馏 (GKD 范式)")
trainer.train(resume_from_checkpoint=False)

# 保存最终模型
print("\n 训练完成，正在保存模型...")
student.save_pretrained("qwen_student_gen_opd")
tokenizer.save_pretrained("qwen_student_gen_opd")
print(" 模型已保存至 qwen_student_gen_opd 文件夹！")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.1: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.8.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
==((====))==  Unsloth 2026.8.1: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


 成功加载本地数据集，数据量: 2000 条
【数据抽样检查 (应为不包含答案的 Prompt)】:
 Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Input:


### Response:

--------------------------------------------------


Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/2000 [00:00<?, ? examples/s]


 开始纯 KL 在线蒸馏 (GKD 范式)


Step,Training Loss
10,5.112227
20,3.888764
30,3.685068
40,3.570179
50,3.498848
60,3.421592
70,3.226230
80,3.362875
90,3.387035
100,3.298434



 训练完成，正在保存模型...
 模型已保存至 qwen_student_gen_opd 文件夹！


下面来计算与教师模型的kl散度

In [1]:
%run code/losses.py

In [2]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer
import warnings
import transformers

# from losses import compute_fkl, compute_rkl
from datasets import load_from_disk

# 强行关闭 HuggingFace 各种警告
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

max_seq_length = 2048
load_in_4bit = True
max_new_tokens = 128
temperature = 2.0


class OPDTrainer(SFTTrainer):
    def __init__(self, *args, teacher_model=None, max_new_tokens=128, temp=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.max_new_tokens = max_new_tokens
        self.temp = temp
        self.tokenizer = kwargs.get("processing_class", kwargs.get("tokenizer"))

    def _generate_on_policy(self, model, input_ids, attention_mask):
        was_training = model.training
        model.eval()
        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                top_p=0.9,
                temperature=0.8,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                use_cache=True,
            )
        if was_training:
            model.train()
        return generated_ids

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        prompt_input_ids = inputs["input_ids"]
        prompt_attention_mask = inputs.get("attention_mask", prompt_input_ids.ne(self.tokenizer.pad_token_id).long())
        prompt_lengths = prompt_attention_mask.sum(dim=1)

        generated_ids = self._generate_on_policy(model, prompt_input_ids, prompt_attention_mask)
        generated_attention_mask = generated_ids.ne(self.tokenizer.pad_token_id).long()

        labels = generated_ids.clone()
        for row_idx, prompt_len in enumerate(prompt_lengths):
            labels[row_idx, :prompt_len] = -100
        labels = labels.masked_fill(generated_attention_mask.eq(0), -100)

        outputs_student = model(
            input_ids=generated_ids,
            attention_mask=generated_attention_mask,
            return_dict=True
        )
        logits = outputs_student.logits

        with torch.no_grad():
            teacher_outputs = self.teacher_model(
                input_ids=generated_ids,
                attention_mask=generated_attention_mask,
                return_dict=True
            )
            teacher_logits = teacher_outputs.logits

        kl = 0
        if isinstance(logits, torch.Tensor) and isinstance(teacher_logits, torch.Tensor):
            if logits.shape[-1] != teacher_logits.shape[-1]:
                teacher_logits = teacher_logits[:, :, :logits.shape[-1]]
            kl = compute_rkl(logits, teacher_logits, labels, padding_id=-100, temp=self.temp)
            valid_tokens = labels.ne(-100).sum().clamp_min(1)
            kl = kl / valid_tokens

        loss_total = kl
        return (loss_total, outputs_student) if return_outputs else loss_total


# 加载模型与数据
print("1. 正在加载已训练完成的 Student 模型 (qwen_student_gen_opd)...")
student, tokenizer = FastLanguageModel.from_pretrained(
    model_name="qwen_student_gen_opd", # 直接加载刚保存的模型
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)
# 评估模式
FastLanguageModel.for_inference(student)
student.eval()

print("2. 正在加载 Teacher 模型 (qwen_teacher_finetune)...")
teacher, _ = FastLanguageModel.from_pretrained(
    model_name="qwen_teacher_finetune",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(teacher)
teacher.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("3. 加载测试集...")
test_dataset = load_from_disk("./data_splits/data_test")
print(f" 测试集加载成功，共 {len(test_dataset)} 条数据")


# 启动评估

args = TrainingArguments(
    output_dir='./eval_results',
    per_device_eval_batch_size=2,
    report_to="none"
)

trainer = OPDTrainer(
    model=student,
    teacher_model=teacher,
    processing_class=tokenizer,
    train_dataset=test_dataset,   # 骗过 Unsloth 的底层检查
    eval_dataset=test_dataset,    # 传入测试集
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=args,
    max_new_tokens=max_new_tokens,
    temp=temperature,
)

print("\n 开始在测试集上计算 KL 散度...")
metrics = trainer.evaluate()

print("\n" + "="*50)
print(f" 最终评估结果 (Test Set KL Divergence): {metrics['eval_loss']:.4f}")
print("="*50)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
1. 正在加载已训练完成的 Student 模型 (qwen_student_gen_opd)...
==((====))==  Unsloth 2026.8.2: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2. 正在加载 Teacher 模型 (qwen_teacher_finetune)...
==((====))==  Unsloth 2026.8.2: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

3. 加载测试集...
 测试集加载成功，共 200 条数据


Unsloth: Tokenizing ["text"] (num_proc=20):   0%|          | 0/200 [00:00<?, ? examples/s]


 开始在测试集上计算 KL 散度...
{'eval_loss': '0.2022', 'eval_model_preparation_time': '0.0378', 'eval_runtime': '479.9', 'eval_samples_per_second': '0.417', 'eval_steps_per_second': '0.208', 'epoch': 0}

 最终评估结果 (Test Set KL Divergence): 0.2022
